# 慢控数据读取与可视化教程

本教程演示如何使用 `sc_reader` 库读取 MySQL 慢控数据并进行可视化分析。

## 目录
1. [环境设置](#环境设置)
2. [数据库探索](#数据库探索)
3. [数据查询](#数据查询)
4. [数据可视化](#数据可视化)
5. [在 Python 脚本中使用](#在-Python-脚本中使用)
6. [完整工作流示例](#完整工作流示例)

## 环境设置

首先导入必要的库并创建数据库连接。

In [2]:
%load_ext autoreload
%autoreload 2

# 导入库
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 导入 sc_reader 库
from sc_reader import SCReader
from sc_reader import visualizer as viz

# 设置 matplotlib 参数
plt.rcParams["axes.unicode_minus"] = False  # 用来正常显示负号

print("导入成功！")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
导入成功！


In [7]:
# 创建数据库连接
# 注意：请根据实际情况修改连接参数
reader = SCReader(host="10.18.154.24", user="read", password="111111", database="slowcontroldata", port=3306)

print("数据库连接成功！")

数据库连接成功！


## 数据库探索

在查询数据之前，先探索数据库中有哪些表和字段。

### 步骤 1：查看所有表

In [8]:
# 列出数据库中所有的表
tables = reader.list_tables()

print(f"数据库中共有 {len(tables)} 个表：")
for i, table in enumerate(tables, 1):
    print(f"{i}. {table}")

数据库中共有 15 个表：
1. daqdata
2. old_piddata250320-250401
3. old_piddata250401-250730
4. old_piddata250731-251128
5. old_runlidata250320-250414
6. old_runlidata250415-251128
7. old_statedata250414-251128
8. old_tempdata250415-250617
9. old_tempdata250617-250711
10. old_tempdata250724-251128
11. old_tempdata251211-260107
12. piddata
13. runlidata
14. statedata
15. tempdata


### 步骤 2：选择一个表并查看表结构

In [9]:
# 选择要查询的表（请根据实际情况修改表名）
if len(tables) == 0:
    raise RuntimeError("数据库中没有可用表，请先检查连接和权限")

table_name = tables[0]  # 使用第一个表作为示例

print(f"选择的表: {table_name}")
print()
print("表结构信息：")

# 查看表结构
table_info = reader.get_table_info(table_name)
print(table_info)
display(table_info)


选择的表: daqdata

表结构信息：
       Field      Type Null Key Default Extra
0  timestamp  datetime  YES        None      
1  data_rate     float  YES        None      


,Field,Type,Null,Key,Default,Extra
0,timestamp,datetime,YES,,None,
1,data_rate,float,YES,,None,


### 预览主要表的数据

预览 piddata, runlidata, statedata, tempdata 这几个主要表的数据。


In [10]:
# 要预览的表名列表
tables_to_preview = ["piddata", "runlidata", "statedata", "tempdata"]

# 预览每个表的数据
for table_name in tables_to_preview:
    if table_name in tables:
        print("=" * 80)
        print(f"Table: {table_name}")
        print("=" * 80)

        # 获取表结构
        try:
            table_info = reader.get_table_info(table_name)
            print(f"\nTable Structure ({len(table_info)} columns):")
            print(table_info)
        except Exception as e:
            print(f"Error getting table info: {e}")
            continue

        # 获取时间范围
        try:
            time_range = reader.get_time_range(table_name)
            print(f"\nTime Range:")
            print(f"  Min: {time_range['min_time']}")
            print(f"  Max: {time_range['max_time']}")
        except Exception as e:
            print(f"Error getting time range: {e}")

        # 预览数据（前5行）
        try:
            preview_table_data = reader.preview_table_data(table_name, limit=5)
            print(f"\nPreview Data (first 5 rows):")
            print(preview_table_data)
            print(f"\nData shape: {preview_table_data.shape}")
        except Exception as e:
            print(f"Error previewing data: {e}")

        print("\n")
    else:
        print(f"Table '{table_name}' not found in database.")
        print("\n")


Table: piddata

Table Structure (9 columns):
           Field         Type Null  Key Default           Extra
0             id          int   NO  PRI    None  auto_increment
1      timestamp  varchar(50)  YES         None                
2  A_Temperature        float  YES         None                
3       A_Heater        float  YES         None                
4  B_Temperature        float  YES         None                
5       B_Heater        float  YES         None                
6  C_Temperature        float  YES         None                
7  D_Temperature        float  YES         None                
8  E_Temperature        float  YES         None                

Time Range:
  Min: 2025-12-11 16:37:51
  Max: 2026-04-17 16:42:48

Preview Data (first 5 rows):
                           id  A_Temperature  A_Heater  B_Temperature  \
timestamp                                                               
2025-12-11 16:37:51+08:00   1        294.638       0.0        294.407   

### 步骤 3：查看表中的列名和数据时间范围

In [11]:
# 获取所有列名
columns = reader.get_table_info(table_name, columns_only=True)
print(f"表 '{table_name}' 包含以下列：")
for i, col in enumerate(columns, 1):
    print(f"{i}. {col}")

# 保存columns变量供后续使用
print(f"\n共 {len(columns)} 个列")

表 'tempdata' 包含以下列：
1. id
2. timestamp
3. Temperature
4. Temperature2
5. Temperature3
6. Temperature4
7. Temperature5
8. Temperature6
9. Temperature7
10. Temperature8

共 10 个列


In [12]:
tables_to_preview

['piddata', 'runlidata', 'statedata', 'tempdata']

In [13]:
# 获取数据的时间范围
time_range = reader.get_time_range(tables_to_preview[0])
print(f"数据时间范围：")
print(f"最早时间: {time_range['min_time']}")
print(f"最晚时间: {time_range['max_time']}")

数据时间范围：
最早时间: 2025-12-11 16:37:51
最晚时间: 2026-04-17 16:42:48


### 步骤 4：预览数据（前几行）

In [14]:
# 预览前 5 行数据
sample_data = reader.preview_table_data(tables_to_preview[0], limit=5)
print(f"表 '{table_name}' 的示例数据：")
print(sample_data)
display(sample_data)

表 'tempdata' 的示例数据：
                           id  A_Temperature  A_Heater  B_Temperature  \
timestamp                                                               
2025-12-11 16:37:51+08:00   1        294.638       0.0        294.407   
2025-12-11 16:38:13+08:00   2        294.638       0.0        294.408   
2025-12-11 16:38:23+08:00   3        294.639       0.0        294.409   
2025-12-11 16:41:15+08:00   4        294.639       0.0        294.413   
2025-12-11 16:41:25+08:00   5        294.639       0.0        294.414   

                           B_Heater  C_Temperature  D_Temperature  \
timestamp                                                           
2025-12-11 16:37:51+08:00       0.0        293.869        293.985   
2025-12-11 16:38:13+08:00       0.0        293.869        293.986   
2025-12-11 16:38:23+08:00       0.0        293.869        293.986   
2025-12-11 16:41:15+08:00       0.0        293.874        294.019   
2025-12-11 16:41:25+08:00       0.0        293.875    

,id,A_Temperature,A_Heater,B_Temperature,B_Heater,C_Temperature,D_Temperature,E_Temperature
timestamp,,,,,,,,
2025-12-11 16:37:51+08:00,1,294.638,0.0,294.407,0.0,293.869,293.985,293.896
2025-12-11 16:38:13+08:00,2,294.638,0.0,294.408,0.0,293.869,293.986,293.899
2025-12-11 16:38:23+08:00,3,294.639,0.0,294.409,0.0,293.869,293.986,293.900
2025-12-11 16:41:15+08:00,4,294.639,0.0,294.413,0.0,293.874,294.019,293.914
2025-12-11 16:41:25+08:00,5,294.639,0.0,294.414,0.0,293.875,294.022,293.914


## 数据查询

演示如何按时间范围查询数据。

### 示例 1：查询指定时间范围的所有数据

In [15]:
tables

['daqdata',
 'old_piddata250320-250401',
 'old_piddata250401-250730',
 'old_piddata250731-251128',
 'old_runlidata250320-250414',
 'old_runlidata250415-251128',
 'old_statedata250414-251128',
 'old_tempdata250415-250617',
 'old_tempdata250617-250711',
 'old_tempdata250724-251128',
 'old_tempdata251211-260107',
 'piddata',
 'runlidata',
 'statedata',
 'tempdata']

In [16]:
# 查询数据（使用实际的时间范围）
# time_range 变量在 cell-14 中已获取
start_time = str(time_range["min_time"])
end_time = str(time_range["max_time"])

data = reader.query_df(table_name=table_name, start_time=start_time, end_time=end_time)

data
# print(f"查询时间范围: {start_time} 到 {end_time}")
# print(f"查询到 {len(data)} 条数据")
# print("数据基本信息：")
# print(data.info())
# print("数据统计摘要：")
# display(data.describe())


,id,Temperature,Temperature2,Temperature3,Temperature4,Temperature5,Temperature6,Temperature7,Temperature8
timestamp,,,,,,,,,
2026-01-07 13:52:48+08:00,1,290.65,290.55,291.05,291.05,290.85,290.95,290.95,290.95
2026-01-07 13:52:51+08:00,2,290.75,290.55,290.95,291.05,290.85,290.85,291.05,290.95
2026-01-07 13:52:54+08:00,3,290.75,290.55,290.95,291.05,290.95,290.85,291.05,290.95
2026-01-07 14:04:00+08:00,4,290.75,290.55,290.95,291.05,290.85,290.85,291.05,290.95
2026-01-07 14:04:03+08:00,5,290.75,290.55,291.05,291.15,290.85,290.85,291.05,291.05
...,...,...,...,...,...,...,...,...,...
2026-04-17 16:42:23+08:00,1671647,162.35,162.55,125.35,126.25,128.75,135.85,131.05,131.15
2026-04-17 16:42:28+08:00,1671648,162.35,162.55,125.35,126.35,128.75,135.75,131.05,131.15
2026-04-17 16:42:33+08:00,1671649,162.35,162.55,125.35,126.35,128.85,135.85,131.05,131.15


### 示例 2：查询特定字段

In [ ]:
# 只查询温度、压力、流量字段（请根据实际列名修改）

# 确保columns变量存在（如果前面的cell没有运行，这里会重新获取）
if "columns" not in locals():
    columns = reader.get_table_info(table_name, columns_only=True)

# 获取时间列名并排除它（时间列会被设置为索引，不应该在查询列中）
try:
    time_col = reader._get_time_column(table_name)
except:
    time_col = None

# 从表中选择要查询的列，排除时间列和id列
exclude_cols = ["id"]
if time_col:
    exclude_cols.append(time_col)
selected_columns = [col for col in columns if col.lower() not in [c.lower() for c in exclude_cols]]
selected_columns = selected_columns[:3]  # 选择前3个数据列作为示例

print(f"查询以下字段: {selected_columns}")

# 使用实际的时间范围
data_selected = reader.query_df(
    table_name=table_name, start_time=start_time, end_time=end_time, columns=selected_columns
)

print(f"\n查询结果的列: {list(data_selected.columns)}")
print(f"查询到 {len(data_selected)} 条数据")
display(data_selected.head())

### 示例 3：查询更精确的时间范围

In [ ]:
# 查询特定日期的特定时间段
# 使用数据范围内的时间进行演示
from datetime import datetime, timedelta

# 解析时间范围
min_time = pd.to_datetime(time_range["min_time"])
max_time = pd.to_datetime(time_range["max_time"])

# 计算一个较短的时间段用于演示（数据范围的前3天）
precise_start = min_time
precise_end = min(min_time + timedelta(days=3), max_time)

data_precise = reader.query_df(table_name=table_name, start_time=str(precise_start), end_time=str(precise_end))

print(f"查询时间范围: {precise_start} 到 {precise_end}")
print(f"查询到 {len(data_precise)} 条数据")
if len(data_precise) > 0:
    print(f"实际数据时间范围: {data_precise.index.min()} 到 {data_precise.index.max()}")

## 多表时间匹配与长表转换

演示如何将多个表按时间匹配对齐，并转换为长表格式进行可视化。


### 步骤 1：从多个表查询数据


In [ ]:
# 导入时间对齐函数
from sc_reader import align_asof

# 选择要查询的多个表（根据实际情况修改）
tables_to_query = ["piddata", "tempdata", "runlidata", "statedata"]

# 查询时间范围（使用实际数据的时间范围）
# 选择一个较短的时间段用于演示
from datetime import datetime, timedelta

# 获取第一个表的时间范围作为参考
if "time_range" not in locals():
    time_range = reader.get_time_range("piddata")

min_time = pd.to_datetime(time_range["min_time"])
max_time = pd.to_datetime(time_range["max_time"])

# 选择前3天作为演示时间段
query_start = min_time
query_end = max_time

print(f"查询时间范围: {query_start} 到 {query_end}")

# 从每个表查询数据
frames = {}
for table in tables_to_query:
    if table in reader.list_tables():
        try:
            print(f"正在查询表: {table}...")
            df = reader.query_df(table_name=table, start_time=str(query_start), end_time=str(query_end))
            if df is not None and len(df) > 0:
                frames[table] = df
                print(f"  ✓ 成功查询 {len(df)} 条数据")
            else:
                print(f"  ⚠ 表 {table} 在指定时间范围内无数据")
        except Exception as e:
            print(f"  ✗ 查询表 {table} 时出错: {e}")
    else:
        print(f"  ⚠ 表 {table} 不存在，跳过")

print(f"\n成功查询 {len(frames)} 个表的数据")


### 步骤 2：按时间匹配对齐多个表


In [ ]:
# 使用 align_asof 按时间匹配对齐多个表
# anchor: 指定以哪个表的时间轴为基准
# tolerance: 时间容差，超出此范围的匹配将填充 NaN
# direction: 匹配方向 ('backward': 向后查找, 'forward': 向前查找, 'nearest': 最近)

if len(frames) == 0:
    raise RuntimeError("没有可对齐的数据，请先检查查询时间范围或表名")

anchor_table = list(frames.keys())[0]
print(f"使用 '{anchor_table}' 作为时间基准表")

# 执行时间对齐
aligned_df = align_asof(
    frames=frames,
    anchor=anchor_table,
    tolerance="200ms",  # 200毫秒的时间容差
    direction="backward",  # 向后查找（找历史最近的）
)

aligned_df


In [ ]:
# 使用 AlignedData 对齐并缓存数据（示例）
from sc_reader import AlignedData, TableSpec

# 创建缓存（time_col 留空时会自动检测）
cache = AlignedData(
    reader,
    [
        TableSpec("tempdata"),
        TableSpec("runlidata"),
        TableSpec("statedata"),
        TableSpec("piddata"),
    ],
    anchor="piddata",
)
new_rows = cache.update(force_full=True)
print(f"首次加载完成，新增 {new_rows} 行")


In [ ]:
new_rows = cache.update(force_full=False)
print(f"本次加载了 {new_rows} 行新数据")

In [ ]:
matched_cols

In [ ]:
# 使用自定义的viz函数进行可视化分析

from sc_reader import plot_temp_pressure_sync
from sc_reader.visualizer import plot_timeseries

# 后续：增量（可能 new_rows=0，但 cache.data 仍然是全量累积）
new_rows = cache.update(force_full=False)
df_all = cache.data  # 累计全量


mask = df_all.columns.str.contains(r"(temp|temperature)", case=False, regex=True) & ~df_all.columns.str.contains(
    r"id", case=False, regex=True
)

matched_cols = df_all.columns[mask].tolist()


# 直接用宽表格式用viz处理
# 自动识别包含温度/压力变量并画图（自己可定义pattern）
plot_timeseries(df_all, column=matched_cols, title="Temperature (all in one)", max_points=int(1e5))


In [ ]:
pressure_cols = [
    c for c in df_all.columns if "id" not in c.lower() and ("press" in c.lower() or "pressure" in c.lower())
]

if len(pressure_cols) > 0:
    plot_timeseries(df_all, column=pressure_cols, title="Pressure (all in one)", max_points=int(1e5))
else:
    print("未找到压力相关列")


### 步骤 3：将宽表转换为长表格式


In [ ]:
# 将宽表转换为长表格式
# 长表格式：每行代表一个时间点的一个变量值
# 列：timestamp, variable, value

if "aligned_df" in locals() and len(aligned_df) > 0:
    # 重置索引，将时间列变为普通列
    long_df = aligned_df.reset_index()

    # 获取时间列名（通常是第一列）
    time_col = long_df.columns[0]

    # 使用 melt 将宽表转换为长表
    # id_vars: 保留的列（时间列）
    # value_vars: 要转换的列（所有数据列）
    # var_name: 新列名（变量名）
    # value_name: 新列名（变量值）
    long_df = pd.melt(
        long_df,
        id_vars=[time_col],
        value_vars=[col for col in long_df.columns if col != time_col],
        var_name="variable",
        value_name="value",
    )

    # 重命名时间列
    long_df = long_df.rename(columns={time_col: "timestamp"})

    # 移除缺失值（可选）
    # long_df = long_df.dropna(subset=['value'])

    print(f"长表数据形状: {long_df.shape}")
    print(f"长表列名: {list(long_df.columns)}")
    print(f"\n长表数据预览（前10行）:")
    display(long_df.head(10))
    print(f"\n变量统计:")
    print(f"  唯一变量数: {long_df['variable'].nunique()}")
    print(f"  变量列表（前10个）: {list(long_df['variable'].unique())[:10]}")
    print(f"\n数据统计:")
    print(long_df.describe())
else:
    print("没有对齐后的数据可供转换")


In [ ]:
# 可视化长表数据：温度画一起，压力画一起（两张图）

import re
import pandas as pd
import matplotlib.pyplot as plt

if "long_df" in locals() and len(long_df) > 0:
    df = long_df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    # 根据 variable 名称进行分组（可按你的命名规则调整）
    temp_mask = df["variable"].str.contains(r"temp|temperature", case=False, na=False)
    press_mask = df["variable"].str.contains(r"press|pressure", case=False, na=False)

    def plot_group(df_sub: pd.DataFrame, title: str):
        if df_sub.empty:
            print(f"{title}: 没有匹配到变量")
            return

        wide = df_sub.pivot_table(index="timestamp", columns="variable", values="value", aggfunc="first").sort_index()

        ax = wide.plot(figsize=(16, 6), linewidth=1)
        ax.set_title(title)
        ax.set_xlabel("timestamp")
        ax.set_ylabel("value")
        ax.grid(True, alpha=0.3)
        plt.legend(loc="best")
        plt.tight_layout()
        plt.show()

        print(f"{title}: {len(wide)} 行, {len(wide.columns)} 个变量")

    plot_group(df[temp_mask], "Temperature (all together)")
    plot_group(df[press_mask], "Pressure (all together)")
else:
    print("没有长表数据可供可视化")


## 数据可视化

使用 `sc_reader.visualizer` 模块提供的函数进行数据可视化。

**注意**: 以下示例使用的列名需要根据你的实际数据进行修改。

### 示例 1：单变量时间序列图

## 在 Python 脚本中使用

以下展示如何在普通 Python 脚本中使用 `sc_reader` 库。

### 完整的 Python 脚本示例

```python
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
慢控数据分析脚本示例
"""

from sc_reader import SCReader
from sc_reader.visualizer import plot_timeseries, plot_correlation
import matplotlib.pyplot as plt

def main():
    # 创建数据库连接
    reader = SCReader(
        host='10.11.50.141',
        user='read',
        password='111111'
    )
    
    # 查询数据
    data = reader.query_by_time(
        table_name='your_table_name',
        start_time='2025-01-01',
        end_time='2025-01-31',
        columns=['temperature', 'pressure', 'flow']
    )
    
    # 生成时间序列图并保存
    fig1, ax1 = plot_timeseries(data, 'temperature')
    fig1.savefig('temperature_timeseries.png', dpi=300, bbox_inches='tight')
    print('已保存: temperature_timeseries.png')
    
    # 生成相关性热力图并保存
    fig2, ax2 = plot_correlation(data)
    fig2.savefig('correlation_heatmap.png', dpi=300, bbox_inches='tight')
    print('已保存: correlation_heatmap.png')
    
    # 将数据保存为 CSV
    data.to_csv('slowcontrol_data.csv')
    print('已保存: slowcontrol_data.csv')
    
    # 关闭数据库连接
    reader.close()
    print('分析完成！')

if __name__ == '__main__':
    main()
```

### 演示：保存图表和数据

## 交互式绘图（Plotly）

`sc_reader` 的所有时间序列绘图函数现在默认使用 Plotly 渲染交互式图表，支持在 notebook 中拖动、缩放、悬停查看数值等操作。所有函数都可以接受和返回 Plotly figure 对象，便于在同一图上叠加不同的数据。

### 主要特性

- **交互式操作**：拖动、缩放、悬停查看数值
- **自动降采样**：大数据集自动降采样（默认阈值 10000 点）以优化性能
- **图形对象复用**：可以传入已有的 figure 对象在同一图上叠加数据
- **向后兼容**：通过 `backend='matplotlib'` 参数仍可使用 matplotlib


In [ ]:
# 示例 1：基础交互式时间序列绘图
# 使用 plot_timeseries（默认使用 Plotly）

# 查询一些数据用于演示
if "data_selected" in locals() and len(data_selected) > 0:
    # 选择第一个数值列
    column_to_plot = data_selected.select_dtypes(include=[np.number]).columns[0]

    # 绘制交互式图表（默认使用 Plotly）
    fig = viz.plot_timeseries(data_selected, column=column_to_plot, title=f"{column_to_plot} 交互式时间序列图")

    # 在 notebook 中显示（Plotly 自动支持交互）
    fig.show()
else:
    print("请先运行前面的代码获取数据")


In [ ]:
# 示例 3：在同一图上叠加不同数据（复用 figure 对象）

if "data_selected" in locals() and len(data_selected) > 0:
    numeric_cols = data_selected.select_dtypes(include=[np.number]).columns.tolist()

    if len(numeric_cols) >= 2:
        # 创建第一个图形
        fig = viz.plot_timeseries(data_selected, column=numeric_cols[0], title="在同一图上叠加多个数据")

        # 在同一图上添加第二个数据
        fig = viz.plot_timeseries(
            data_selected,
            column=numeric_cols[1],
            fig=fig,  # 传入已有的 figure
            color="coral",  # 使用不同颜色
        )

        fig.show()
        print(f"已在同一图上叠加了 {numeric_cols[0]} 和 {numeric_cols[1]}")
    else:
        print("数据列不足，无法展示叠加功能")
else:
    print("请先运行前面的代码获取数据")


In [ ]:
# 示例 4：通用 plot() 函数自动识别数据并绘制

if "data_selected" in locals() and len(data_selected) > 0:
    from sc_reader import plot_timeseries

    # 自动识别：传入 DataFrame，自动绘制所有数值列
    fig = plot_timeseries(data_selected)
    fig.show()

    print("\n通用 plot() 函数可以：")
    print("- 自动识别 DataFrame/Series")
    print("- 自动选择数值列")
    print("- 自动选择合适的绘图函数")
    print("- 支持单列或多列绘制")
else:
    print("请先运行前面的代码获取数据")


In [ ]:
# 示例 5：自动降采样演示（大数据集优化）

# 创建大量数据点用于演示降采样
if "data_selected" in locals() and len(data_selected) > 0:
    # 假设数据量很大，会自动降采样
    numeric_cols = data_selected.select_dtypes(include=[np.number]).columns.tolist()

    if numeric_cols:
        # 使用 max_points 参数控制降采样阈值
        fig = viz.plot_timeseries(
            data_selected,
            column=numeric_cols[0],
            max_points=5000,  # 自定义降采样阈值（默认 10000）
            title=f"{numeric_cols[0]} (自动降采样优化)",
        )
        fig.show()

        print(f"数据点数量: {len(data_selected)}")
        print("如果超过 max_points，会自动降采样并显示警告信息")
else:
    print("请先运行前面的代码获取数据")


### 示例 2.5：从 cache 长表中绘制所有温度数据

从 AlignedData 获取数据，转换为长表格式，筛选温度相关的变量，并绘制交互式图表。


In [ ]:
cache.update()

In [ ]:
# 从 cache 获取数据并绘制所有温度相关的数据
cache.update()
df_wide = cache.data.copy()

# 直接筛选温度相关的列（匹配 temp 或 temperature，不区分大小写）
import re

temp_columns = [col for col in df_wide.columns if re.search(r"temp|temperature", col, re.IGNORECASE)]

if len(temp_columns) > 0:
    # 筛选温度相关的数据
    df_temp = df_wide[temp_columns]

    print(f"找到 {len(temp_columns)} 个温度相关的变量：")
    for col in temp_columns:
        print(f"  - {col}")

    fig = viz.plot_timeseries(
        df_temp, column=temp_columns, title="所有温度数据（交互式）", ylabel="Temperature", max_points=100000
    )
    # fig.update_yaxes(0,300)
    fig.show()

    print()
    print(f"已绘制 {len(temp_columns)} 个温度变量的时间序列")
    print("提示：可以拖动、缩放、悬停查看数值，点击图例显示/隐藏曲线")
else:
    print("未找到温度相关的变量")
    print(f"可用列：{list(df_wide.columns)[:10]}...")  # 显示前10个列名


In [ ]:
cache.data

In [ ]:
cache.update()

# 自动选择一个温度列进行交互式刷新演示
temp_cols = [c for c in cache.data.columns if "temp" in c.lower() or "temperature" in c.lower()]
if len(temp_cols) == 0:
    raise RuntimeError("cache.data 中未找到温度相关列，无法演示 refresh_plot")

demo_col = temp_cols[0]
print(f"使用列: {demo_col}")

fig = cache.plot_timeseries(demo_col)
fig.show()


In [ ]:
# 用户拖动到感兴趣的时间范围后
fig = cache.refresh_plot(fig)  # 刷新为高分辨率
fig.show()

In [ ]:
# 示例 6：双Y轴交互式图表（压力 vs 温度）- 使用 cache.data

# 检查 cache 是否存在
if "cache" in locals() and hasattr(cache, "data"):
    import re

    # 从 cache.data 获取数据
    df = cache.data.copy()

    if len(df) > 0:
        # 筛选压力相关的列
        pressure_cols = [col for col in df.columns if re.search(r"press|pressure", col, re.IGNORECASE)]

        # 筛选温度相关的列
        temp_cols = [col for col in df.columns if re.search(r"temp|temperature", col, re.IGNORECASE)]

        if len(pressure_cols) > 0 and len(temp_cols) > 0:
            # 左Y轴：压力，右Y轴：温度
            fig = viz.plot_dual_axis(
                df,
                left_column=pressure_cols[0],
                right_column=temp_cols[0],
                title="压力 vs 温度（双Y轴交互式图表）",
                left_ylabel="Pressure",
                right_ylabel="Temperature",
                left_color="steelblue",
                right_color="coral",
            )
            fig.show()

            print(f"数据来源: cache.data")
            print(f"数据点数: {len(df)}")
            print(f"左Y轴（压力）: {pressure_cols[0]}")
            print(f"右Y轴（温度）: {temp_cols[0]}")
            print(f"\n可用压力列: {pressure_cols}")
            print(f"可用温度列: {temp_cols}")
            print("\nPlotly 原生支持双Y轴，两个轴可以独立交互")
        elif len(pressure_cols) == 0:
            print("未找到压力相关的列")
            print(f"可用列：{list(df.columns)[:10]}...")
        elif len(temp_cols) == 0:
            print("未找到温度相关的列")
            print(f"可用列：{list(df.columns)[:10]}...")
    else:
        print("cache 中没有数据，请先运行 cache.update() 加载数据")
else:
    print("cache 未定义，请先创建 AlignedData 并加载数据")
    print("\n示例代码：")
    print("from sc_reader import AlignedData, TableSpec")
    print(
        "cache = AlignedData(reader, [TableSpec('tempdata', 'timestamp'), TableSpec('runlidata', 'timestamp')], anchor='tempdata')"
    )
    print("cache.update(force_full=True)")


In [ ]:
fig = viz.plot_temp_pressure_sync(cache, max_points=10000)
fig.show()

# 用户拖动到感兴趣的时间范围后
fig = cache.refresh_plot(fig)  # 刷新为高分辨率
fig.show()

## 总结

本教程展示了如何使用 `sc_reader` 库进行慢控数据的读取、分析和可视化：

### 主要功能

**数据读取 (SCReader)**
- `list_tables()` - 列出所有表
- `get_table_info()` - 获取表结构
- `query_by_time()` - 按时间范围查询
- `get_time_range()` - 获取数据时间范围

**数据可视化 (visualizer)**
- `plot_timeseries()` - 时间序列图
- `plot_multi_variables()` - 多变量对比
- `plot_dual_axis()` - 双Y轴图表
- `plot_subplots()` - 子图布局
- `plot_distribution()` - 分布直方图
- `plot_boxplot()` - 箱线图
- `plot_correlation()` - 相关性热力图
- `plot_rolling_stats()` - 滚动统计

### 使用建议

1. **数据探索**：使用前先探索数据库结构和时间范围
2. **时间范围**：根据实际需求选择合适的查询时间范围
3. **列选择**：对于大数据量，只查询需要的列可以提高性能
4. **保存结果**：及时保存图表和数据以便后续使用
5. **自定义**：所有可视化函数都返回 fig 和 ax 对象，可以进一步自定义

### 下一步

- 根据实际数据调整列名和参数
- 创建自己的分析脚本
- 探索更多的可视化组合
- 添加数据处理和分析逻辑

In [ ]:
# 记得关闭数据库连接
reader.close()
print("数据库连接已关闭")